# LeetCode 2873 — Maximum Value of an Ordered Triplet I bằng Python

Link đề: https://leetcode.com/problems/maximum-value-of-an-ordered-triplet-i/ · Độ khó: **Easy** · Chủ đề: Array (prefix / suffix maximum)

Skeleton LeetCode:

```python
class Solution:
    def maximumTripletValue(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn ba chỉ số **tăng dần** `i < j < k` để **tối đa hóa** `(nums[i] - nums[j]) * nums[k]`; nếu mọi bộ ba đều cho giá trị âm thì trả `0`.
2. Biết cách **đọc ràng buộc** (`n ≤ 100` cho phép `O(n³)`; `nums[i] ≥ 1` giúp tham lam ở `nums[k]`).
3. Đi từ cách **vét cạn ba vòng lặp** → **hai vòng lặp + suffix max** → **mảng prefix max / suffix max** (`O(n)`) → **một lượt duyệt, `O(1)` bộ nhớ** → **bản gọn bằng `itertools.accumulate`**.
4. Hiểu kỹ thuật **"cố định phần tử giữa `j`, lấy tốt nhất bên trái và bên phải"**: một mẫu rất hay gặp.
5. Tránh các lỗi kinh điển: quên `max(0, …)`, dùng `max` / `min` toàn mảng không tôn trọng thứ tự, prefix / suffix lấy luôn cả `j`, khởi tạo đáp án bằng `-inf`.

## 2. Đề bài & ví dụ

Cho một mảng số nguyên **đánh chỉ số từ 0** `nums`.

Trả về **giá trị lớn nhất** của mọi **bộ ba có thứ tự** `(i, j, k)` với `i < j < k`. Giá trị của bộ ba là

`(nums[i] - nums[j]) * nums[k]`.

Nếu **mọi** bộ ba đều có giá trị **âm** thì trả về `0`.

**Ví dụ 1**

- Input: `nums = [12, 6, 1, 2, 7]`
- Output: `77`
- Giải thích: bộ ba `(0, 2, 4)` có giá trị `(12 - 1) * 7 = 77`; không có bộ ba nào cho giá trị lớn hơn.

**Ví dụ 2**

- Input: `nums = [1, 10, 3, 4, 19]`
- Output: `133`
- Giải thích: bộ ba `(1, 2, 4)` có giá trị `(10 - 3) * 19 = 133`.

**Ví dụ 3**

- Input: `nums = [1, 2, 3]`
- Output: `0`
- Giải thích: bộ ba duy nhất `(0, 1, 2)` có giá trị `(1 - 2) * 3 = -3 < 0`, nên trả `0`.

**Constraints**

- `3 <= nums.length <= 100`
- `1 <= nums[i] <= 10^6`

In [1]:
from typing import List
nums = [12, 6, 1, 2, 7]
print((nums[0] - nums[2]) * nums[4])     # 77  (bộ ba (0, 2, 4))
print((nums[0] - nums[1]) * nums[4])     # 42  (bộ ba (0, 1, 4)) - nhỏ hơn

77
42


## 3. Kiến thức nền

### 3.1. Bộ ba chỉ số `i < j < k` bằng ba vòng `for`

Để liệt kê mọi bộ ba có thứ tự: `i` chạy `0..n-1`, `j` chạy từ `i + 1`, `k` chạy từ `j + 1`. Cách viết này tự đảm bảo `i < j < k` và không lặp bộ ba nào.

In [2]:
n = 4
triples = []
for i in range(n):
    for j in range(i + 1, n):
        for k in range(j + 1, n):
            triples.append((i, j, k))
print(triples)       # [(0,1,2), (0,1,3), (0,2,3), (1,2,3)]  -> C(4,3) = 4 bộ

[(0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3)]


### 3.2. `itertools.combinations`

`combinations(range(n), 3)` sinh đúng các bộ ba tăng dần như trên, ngắn hơn ba vòng lặp.

In [3]:
from itertools import combinations
print(list(combinations(range(4), 3)))
print(len(list(combinations(range(100), 3))))     # 161700 = C(100, 3): số bộ ba tối đa của đề

[(0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3)]
161700


### 3.3. Tích với số âm và `max(0, x)`

Hiệu `nums[i] - nums[j]` có thể **âm** (khi `nums[i] < nums[j]`). `nums[k]` luôn **dương** (theo ràng buộc `≥ 1`). Vậy giá trị bộ ba âm ⇔ hiệu âm; bằng 0 ⇔ hiệu bằng 0. Hàm `max(0, x)` chặn kết quả âm thành `0`.

In [4]:
print((1 - 2) * 3)             # -3
print(max(0, (1 - 2) * 3))     # 0
print(max(0, (10 - 3) * 19))   # 133
print(max([-3, -5, -1], default=0))   # -1 (max vẫn trả số âm nếu list có số âm)

-3
0
133
-1


### 3.4. Prefix max và suffix max

- **Prefix max** `pre[j]` = giá trị lớn nhất trong `nums[0..j-1]` (các phần tử **trước** `j`).
- **Suffix max** `suf[j]` = giá trị lớn nhất trong `nums[j+1..n-1]` (các phần tử **sau** `j`).

Dựng một lần, rồi tra cứu `O(1)`.

In [5]:
nums = [12, 6, 1, 2, 7]
n = len(nums)
pre = [0] * n          # pre[0] không dùng (chưa có phần tử nào trước)
cur = nums[0]
for j in range(1, n):
    pre[j] = cur                      # max của nums[0..j-1]
    cur = max(cur, nums[j])
suf = [0] * n          # suf[n-1] không dùng
cur = nums[-1]
for j in range(n - 2, -1, -1):
    suf[j] = cur                      # max của nums[j+1..n-1]
    cur = max(cur, nums[j])
print("pre =", pre)    # [0, 12, 12, 12, 12]
print("suf =", suf)    # [7, 7, 7, 7, 0]

pre = [0, 12, 12, 12, 12]
suf = [7, 7, 7, 7, 0]


### 3.5. `itertools.accumulate` với `max`

`accumulate(nums, max)` trả về dãy "max đến hiện tại": `[12, 12, 12, 12, 12]` cho ví dụ trên. Có thể dùng để dựng prefix max ngắn gọn.

In [6]:
from itertools import accumulate
print(list(accumulate([12, 6, 1, 2, 7], max)))     # [12, 12, 12, 12, 12]
print(list(accumulate([1, 10, 3, 4, 19], max)))    # [1, 10, 10, 10, 19]
print(list(accumulate([1, 10, 3, 4, 19][::-1], max))[::-1])   # suffix max: [19, 19, 19, 19, 19]

[12, 12, 12, 12, 12]
[1, 10, 10, 10, 19]
[19, 19, 19, 19, 19]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ta muốn `(nums[i] - nums[j]) * nums[k]` **lớn nhất**. Nhìn công thức:

- `nums[k]` luôn dương, nên với `i, j` cố định, ta muốn `nums[k]` **càng lớn càng tốt**.
- Hiệu `nums[i] - nums[j]`: với `j` cố định ta muốn `nums[i]` **càng lớn càng tốt**.

Hai lựa chọn **độc lập nhau** khi đã cố định `j`: `i` chỉ cần nằm bên trái `j`, `k` chỉ cần nằm bên phải `j`.

Ví dụ 2: `nums = [1, 10, 3, 4, 19]`. Thử `j = 2` (giá trị 3):

- bên trái `j`: `{1, 10}`, lớn nhất là `10`;
- bên phải `j`: `{4, 19}`, lớn nhất là `19`;
- giá trị = `(10 - 3) * 19 = 133`.

Thử `j = 1` (giá trị 10): bên trái chỉ có `1`, hiệu `1 - 10 = -9 < 0` — cho giá trị âm, bỏ.

## 5. Cách 1 — Brute Force / baseline: ba vòng lặp

Thử mọi bộ ba `i < j < k`, tính giá trị, giữ lớn nhất. Khởi tạo đáp án `0` để tự động xử lý trường hợp "mọi bộ ba đều âm".

In [7]:
def triplet_bruteforce(nums: List[int]) -> int:
    n = len(nums)
    best = 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                best = max(best, (nums[i] - nums[j]) * nums[k])
    return best

print(triplet_bruteforce([12, 6, 1, 2, 7]))     # 77
print(triplet_bruteforce([1, 10, 3, 4, 19]))    # 133
print(triplet_bruteforce([1, 2, 3]))            # 0

# phiên bản combinations (tương đương)
def triplet_combinations(nums: List[int]) -> int:
    return max([0] + [(nums[i] - nums[j]) * nums[k] for i, j, k in combinations(range(len(nums)), 3)])

print(triplet_combinations([12, 6, 1, 2, 7]))   # 77

77
133
0
77


**Độ phức tạp:** có `C(n, 3)` bộ ba nên **Time `O(n³)`**, **Space `O(1)`**. Với `n ≤ 100` chỉ cỡ `161 700` bộ: **vẫn đủ nhanh** để qua bài (đó là lý do bài được xếp Easy).

**Nút thắt:** ta thử cả những `k` chắc chắn thua. Với `i, j` cố định, chỉ `k` có `nums[k]` lớn nhất bên phải `j` mới đáng xét.

## 6. Ý tưởng thuật toán / bước đột phá

**Cố định phần tử giữa `j`.** Với `j` cố định:

- `i` nằm trong `[0, j-1]`, ta muốn `nums[i]` lớn nhất: `pre[j] = max(nums[0..j-1])`.
- `k` nằm trong `[j+1, n-1]`, ta muốn `nums[k]` lớn nhất: `suf[j] = max(nums[j+1..n-1])`.

Giá trị tốt nhất tại `j` là `(pre[j] - nums[j]) * suf[j]`. Đáp án = `max(0, max_j (pre[j] - nums[j]) * suf[j])`.

**Vì sao chọn `suf[j]` lớn nhất luôn tốt?** Nếu hiệu `(pre[j] - nums[j]) > 0` thì nhân với số dương lớn nhất cho tích lớn nhất. Nếu hiệu `≤ 0` thì mọi tích `≤ 0` và đã bị chặn bởi `max(0, …)`; không ảnh hưởng đáp án.

Hai cách tổ chức: (a) dựng mảng `pre`, `suf` (Cách 3), (b) duyệt một lượt, giữ biến chạy (Cách 4).

## 7. Các cách tiếp theo

### Cách 2 — Hai vòng lặp `(i, j)` + suffix max

**Giải thích.** Nếu chỉ cố định `i, j`, ta không cần vòng `k`: `k` tốt nhất là phần tử lớn nhất sau `j`, tức `suf[j]`. Dựng `suf` một lần, rồi chạy hai vòng `i < j`.

In [8]:
def triplet_two_loops(nums: List[int]) -> int:
    n = len(nums)
    suf = [0] * n
    cur = nums[-1]
    for j in range(n - 2, -1, -1):
        suf[j] = cur                # max của nums[j+1..]
        cur = max(cur, nums[j])
    best = 0
    for i in range(n):
        for j in range(i + 1, n - 1):         # j phải còn chỗ cho k, nên j <= n - 2
            best = max(best, (nums[i] - nums[j]) * suf[j])
    return best

print(triplet_two_loops([12, 6, 1, 2, 7]))    # 77
print(triplet_two_loops([1, 10, 3, 4, 19]))   # 133

77
133


In [9]:
# Code nộp LeetCode cho Cách 2
class SolutionTwoLoops:
    def maximumTripletValue(self, nums: List[int]) -> int:
        n = len(nums)
        suf = [0] * n
        cur = nums[-1]
        for j in range(n - 2, -1, -1):
            suf[j] = cur
            cur = max(cur, nums[j])
        best = 0
        for i in range(n):
            for j in range(i + 1, n - 1):
                best = max(best, (nums[i] - nums[j]) * suf[j])
        return best

print(SolutionTwoLoops().maximumTripletValue([1, 2, 3]))   # 0

0


**Độ phức tạp Cách 2:** hai vòng lồng nhau `O(n²)`, dựng `suf` `O(n)`. **Time `O(n²)`**, **Space `O(n)`**.

---

### Cách 3 — Mảng prefix max và suffix max (`O(n)`)

**Giải thích từng đoạn.**

1. `pre[j]` = max các phần tử **trước** `j` (dựng từ trái sang phải).
2. `suf[j]` = max các phần tử **sau** `j` (dựng từ phải sang trái).
3. Với mỗi `j` từ `1` tới `n - 2`: ứng viên `(pre[j] - nums[j]) * suf[j]`.
4. Đáp án là max các ứng viên và `0`.

In [10]:
def triplet_prefix_suffix(nums: List[int]) -> int:
    n = len(nums)
    pre = [0] * n
    cur = nums[0]
    for j in range(1, n):
        pre[j] = cur
        cur = max(cur, nums[j])
    suf = [0] * n
    cur = nums[-1]
    for j in range(n - 2, -1, -1):
        suf[j] = cur
        cur = max(cur, nums[j])
    best = 0
    for j in range(1, n - 1):
        best = max(best, (pre[j] - nums[j]) * suf[j])
    return best

print(triplet_prefix_suffix([12, 6, 1, 2, 7]))     # 77
print(triplet_prefix_suffix([1, 10, 3, 4, 19]))    # 133
print(triplet_prefix_suffix([1, 2, 3]))            # 0

77
133
0


In [11]:
# Code nộp LeetCode cho Cách 3
class SolutionPrefixSuffix:
    def maximumTripletValue(self, nums: List[int]) -> int:
        n = len(nums)
        pre = [0] * n
        cur = nums[0]
        for j in range(1, n):
            pre[j] = cur
            cur = max(cur, nums[j])
        suf = [0] * n
        cur = nums[-1]
        for j in range(n - 2, -1, -1):
            suf[j] = cur
            cur = max(cur, nums[j])
        best = 0
        for j in range(1, n - 1):
            best = max(best, (pre[j] - nums[j]) * suf[j])
        return best

print(SolutionPrefixSuffix().maximumTripletValue([12, 6, 1, 2, 7]))   # 77

77


**Độ phức tạp Cách 3:** ba lượt duyệt đơn. **Time `O(n)`**, **Space `O(n)`** cho hai mảng.

---

### Cách 4 — Một lượt duyệt, `O(1)` bộ nhớ

**Giải thích.** Duyệt `x = nums[t]` từ trái sang phải; xem `x` đóng ba vai theo thứ tự:

1. **Vai `k`:** `best = max(best, max_diff * x)`, với `max_diff` là hiệu `nums[i] - nums[j]` lớn nhất của mọi cặp `i < j < t` đã thấy.
2. **Vai `j`:** `max_diff = max(max_diff, imax - x)`, với `imax` = max các phần tử **trước** `t`.
3. **Vai `i`:** `imax = max(imax, x)`.

Thứ tự 1 → 2 → 3 rất quan trọng: mỗi bước chỉ dùng thông tin của các phần tử **đứng trước** `t`. `max_diff` khởi tạo `0` (chặn âm, vì đáp án cũng được chặn ở `0`).

In [12]:
def triplet_one_pass(nums: List[int]) -> int:
    best = 0
    max_diff = 0              # max(nums[i] - nums[j]) với i < j đã duyệt (chặn dưới 0)
    imax = nums[0]            # max các phần tử đã duyệt
    for x in nums[1:]:
        best = max(best, max_diff * x)        # x làm nums[k]
        max_diff = max(max_diff, imax - x)    # x làm nums[j]
        imax = max(imax, x)                   # x làm nums[i] cho các bước sau
    return best

print(triplet_one_pass([12, 6, 1, 2, 7]))     # 77
print(triplet_one_pass([1, 10, 3, 4, 19]))    # 133
print(triplet_one_pass([1, 2, 3]))            # 0

77
133
0


In [13]:
# Code nộp LeetCode cho Cách 4
class SolutionOnePass:
    def maximumTripletValue(self, nums: List[int]) -> int:
        best = 0
        max_diff = 0
        imax = nums[0]
        for x in nums[1:]:
            best = max(best, max_diff * x)
            max_diff = max(max_diff, imax - x)
            imax = max(imax, x)
        return best

print(SolutionOnePass().maximumTripletValue([1, 10, 3, 4, 19]))   # 133

133


**Độ phức tạp Cách 4:** một lượt duyệt, **Time `O(n)`**, **Space `O(1)`**.

---

### Cách 5 — `itertools.accumulate` (gọn)

**Giải thích.** `accumulate(nums, max)` là prefix max, `accumulate(reversed(nums), max)` là suffix max (đảo lại). Với mỗi `j` ta ghép `pre[j-1]`, `nums[j]`, `suf[j+1]`. Chú ý `pre[j-1]` là max của `nums[0..j-1]` và `suf[j+1]` là max của `nums[j+1..]`.

In [14]:
from itertools import accumulate

def triplet_accumulate(nums: List[int]) -> int:
    n = len(nums)
    pre = list(accumulate(nums, max))                  # pre[t] = max(nums[0..t])
    suf = list(accumulate(reversed(nums), max))[::-1]  # suf[t] = max(nums[t..])
    return max([0] + [(pre[j - 1] - nums[j]) * suf[j + 1] for j in range(1, n - 1)])

print(triplet_accumulate([12, 6, 1, 2, 7]))     # 77
print(triplet_accumulate([1, 10, 3, 4, 19]))    # 133

77
133


In [15]:
# Code nộp LeetCode cho Cách 5
class SolutionAccumulate:
    def maximumTripletValue(self, nums: List[int]) -> int:
        n = len(nums)
        pre = list(accumulate(nums, max))
        suf = list(accumulate(reversed(nums), max))[::-1]
        return max([0] + [(pre[j - 1] - nums[j]) * suf[j + 1] for j in range(1, n - 1)])

print(SolutionAccumulate().maximumTripletValue([1, 2, 3]))   # 0

0


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(n)`**.

## 8. TRACE từng bước

Trace Cách 3 trên `nums = [12, 6, 1, 2, 7]`:

| `j` | `nums[j]` | `pre[j]` (max trước j) | `suf[j]` (max sau j) | `(pre - nums[j]) * suf` | `best` sau |
|---|---|---|---|---|---|
| 1 | 6 | 12 | 7 | `(12-6)*7 = 42` | 42 |
| 2 | 1 | 12 | 7 | `(12-1)*7 = 77` | **77** |
| 3 | 2 | 12 | 7 | `(12-2)*7 = 70` | 77 |

In [16]:
def trace_prefix_suffix(nums):
    n = len(nums)
    pre, cur = [0] * n, nums[0]
    for j in range(1, n):
        pre[j] = cur
        cur = max(cur, nums[j])
    suf, cur = [0] * n, nums[-1]
    for j in range(n - 2, -1, -1):
        suf[j] = cur
        cur = max(cur, nums[j])
    print("nums =", nums)
    print("pre  =", pre)
    print("suf  =", suf)
    best = 0
    for j in range(1, n - 1):
        val = (pre[j] - nums[j]) * suf[j]
        best = max(best, val)
        print(f"j={j}: nums[j]={nums[j]}, ({pre[j]} - {nums[j]}) * {suf[j]} = {val}, best={best}")
    return best

trace_prefix_suffix([12, 6, 1, 2, 7])
trace_prefix_suffix([1, 2, 3])

nums = [12, 6, 1, 2, 7]
pre  = [0, 12, 12, 12, 12]
suf  = [7, 7, 7, 7, 0]
j=1: nums[j]=6, (12 - 6) * 7 = 42, best=42
j=2: nums[j]=1, (12 - 1) * 7 = 77, best=77
j=3: nums[j]=2, (12 - 2) * 7 = 70, best=77
nums = [1, 2, 3]
pre  = [0, 1, 2]
suf  = [3, 3, 0]
j=1: nums[j]=2, (1 - 2) * 3 = -3, best=0


0

In [17]:
def trace_one_pass(nums):
    best, max_diff, imax = 0, 0, nums[0]
    print(f"khởi tạo: best={best}, max_diff={max_diff}, imax={imax}")
    for t, x in enumerate(nums[1:], start=1):
        old = (best, max_diff, imax)
        best = max(best, max_diff * x)
        max_diff = max(max_diff, imax - x)
        imax = max(imax, x)
        print(f"t={t}, x={x}: best {old[0]}->{best} (dùng max_diff={old[1]}), "
              f"max_diff {old[1]}->{max_diff} (imax={old[2]}-{x}), imax {old[2]}->{imax}")
    return best

trace_one_pass([12, 6, 1, 2, 7])

khởi tạo: best=0, max_diff=0, imax=12
t=1, x=6: best 0->0 (dùng max_diff=0), max_diff 0->6 (imax=12-6), imax 12->12
t=2, x=1: best 0->6 (dùng max_diff=6), max_diff 6->11 (imax=12-1), imax 12->12
t=3, x=2: best 6->22 (dùng max_diff=11), max_diff 11->11 (imax=12-2), imax 12->12
t=4, x=7: best 22->77 (dùng max_diff=11), max_diff 11->11 (imax=12-7), imax 12->12


77

## 9. Code nộp LeetCode

In [18]:
from typing import List

class Solution:
    def maximumTripletValue(self, nums: List[int]) -> int:
        best = 0
        max_diff = 0
        imax = nums[0]
        for x in nums[1:]:
            best = max(best, max_diff * x)        # x là nums[k]
            max_diff = max(max_diff, imax - x)    # x là nums[j]
            imax = max(imax, x)                   # x là nums[i] cho các bước sau
        return best

print(Solution().maximumTripletValue([12, 6, 1, 2, 7]))    # 77
print(Solution().maximumTripletValue([1, 10, 3, 4, 19]))   # 133
print(Solution().maximumTripletValue([1, 2, 3]))           # 0

77
133
0


## 10. Vì sao thuật toán đúng

Xét Cách 4 (Cách 3 chứng minh tương tự).

- **Thứ tự hợp lệ:** trước khi xử lý `x` ở chỉ số `t`, `imax = max(nums[0..t-1])` và `max_diff = max(0, max_{i<j<t}(nums[i] - nums[j]))`. Ban đầu (`t = 1`) `imax = nums[0]`, `max_diff = 0` đúng vì chưa có cặp `i < j < 1`.
- **Vai `k`:** với `k = t`, giá trị tốt nhất là `(max_{i<j<t}(nums[i] - nums[j])) * nums[t]` vì `nums[t] > 0` nên nhân với hiệu lớn nhất cho tích lớn nhất. Nếu hiệu lớn nhất là âm thì mọi giá trị bộ ba kết thúc tại `k = t` đều âm, ta dùng `0` (không đổi đáp án vì đáp án ≥ 0).
- **Vai `j`:** sau khi xét `x` là `k`, ta cập nhật `max_diff` với các cặp `(i, j = t)`: hiệu lớn nhất là `max(nums[0..t-1]) - nums[t] = imax - x`.
- **Vai `i`:** cuối cùng cập nhật `imax` để dùng cho các `t` sau.
- **Đầy đủ:** mọi bộ ba `(i, j, k)` có `k = t` nào đó, ở thời điểm đó cả `i` và `j` đã được tính trong `max_diff`, nên không bỏ sót. Cuối cùng `best = max(0, giá trị lớn nhất của mọi bộ ba)`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Ba vòng lặp | `O(n³)` | `O(1)` |
| 2. Hai vòng + suffix max | `O(n²)` | `O(n)` |
| 3. Prefix / suffix max | `O(n)` | `O(n)` |
| 4. Một lượt, biến chạy | `O(n)` | `O(1)` |
| 5. `accumulate` | `O(n)` | `O(n)` |

Với `n = 100`: Cách 1 cỡ `161 700` bộ ba, Cách 3 và 4 chỉ cỡ `100` bước. Giá trị tối đa `(10⁶ - 1) * 10⁶ ≈ 10¹²`: Python không tràn số, nhưng khi làm bằng C++/Java phải dùng `long` / `long long` (vượt 32-bit).

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

In [19]:
sols = [SolutionTwoLoops(), SolutionPrefixSuffix(), SolutionOnePass(), SolutionAccumulate(), Solution()]

cases = [
    ("ví dụ 1", [12, 6, 1, 2, 7], 77),
    ("ví dụ 2", [1, 10, 3, 4, 19], 133),
    ("ví dụ 3", [1, 2, 3], 0),
    ("n = 3 dương", [5, 1, 2], 8),            # (5 - 1) * 2
    ("n = 3 hiệu 0", [4, 4, 9], 0),
    ("toàn bằng nhau", [5, 5, 5, 5], 0),
    ("giảm dần", [10, 5, 1], 5),              # (10 - 5) * 1
    ("tăng dần", [1, 2, 3, 4, 5], 0),
    ("giá trị lớn nhất cuối", [9, 1, 1, 1000000], 8 * 1000000),
    ("đáp án không dùng phần tử lớn nhất", [100, 1, 2, 3, 4], 99 * 4),
    ("nhiều ứng viên", [3, 8, 1, 9, 2, 7], 63),       # (8-1)*9 = 63 > (9-2)*7 = 49
]
for name, nums, exp in cases:
    b = triplet_bruteforce(nums)
    assert b == exp, (name, b, exp)
    for s in sols:
        got = s.maximumTripletValue(list(nums))
        assert got == b, (name, type(s).__name__, got, b)
    print("PASS", name, "->", b)

PASS ví dụ 1 -> 77
PASS ví dụ 2 -> 133
PASS ví dụ 3 -> 0
PASS n = 3 dương -> 8
PASS n = 3 hiệu 0 -> 0
PASS toàn bằng nhau -> 0
PASS giảm dần -> 5
PASS tăng dần -> 0
PASS giá trị lớn nhất cuối -> 8000000
PASS đáp án không dùng phần tử lớn nhất -> 396
PASS nhiều ứng viên -> 63


Kiểm tay ca cuối `[3, 8, 1, 9, 2, 7]`: `j = 2` (giá trị 1) có `pre = 8`, `suf = 9` nên `(8 - 1) * 9 = 63`; `j = 4` (giá trị 2) có `pre = 9`, `suf = 7` nên `(9 - 2) * 7 = 49`. Lớn nhất là `63`.

### 12.2. Phản ví dụ cho các lỗi hay gặp

Mảng mà `max(nums)` và `min(nums)` nằm sai thứ tự: lấy `(max - min) * max` bỏ qua `i < j < k` sẽ ra giá trị sai.

In [20]:
nums = [1, 2, 3]                     # (max - min) * max = (3 - 1) * 3 = 6 nhưng đúng là 0
naive = (max(nums) - min(nums)) * max(nums)
print("sai :", naive)
print("đúng:", Solution().maximumTripletValue(nums))
assert Solution().maximumTripletValue(nums) == 0 != naive
nums = [9, 1, 5]                     # (9 - 1) * 5 = 40
assert Solution().maximumTripletValue(nums) == 40
print("PASS phản ví dụ thứ tự")

sai : 6
đúng: 0
PASS phản ví dụ thứ tự


### 12.3. Đối chiếu ngẫu nhiên (có seed) đúng ràng buộc

`n ∈ [3, 100]`, `nums[i] ∈ [1, 10⁶]`. Brute force `O(n³)` hơi chậm với `n = 100`, nên ta chia: `n ≤ 25` so với brute force; `n` lớn so với Cách 2 (`O(n²)`, rất tin cậy).

In [21]:
import random
random.seed(2873)
for t in range(400):
    n = random.randint(3, 25)
    hi = random.choice([5, 30, 10**6])
    nums = [random.randint(1, hi) for _ in range(n)]
    exp = triplet_bruteforce(nums)
    for s in sols:
        assert s.maximumTripletValue(list(nums)) == exp, (t, type(s).__name__, nums)
print("400 test nhỏ: mọi cách khớp brute force")

for t in range(300):
    n = random.randint(26, 100)
    hi = random.choice([5, 100, 10**6])
    nums = [random.randint(1, hi) for _ in range(n)]
    exp = triplet_two_loops(nums)
    for s in sols:
        assert s.maximumTripletValue(list(nums)) == exp, (t, type(s).__name__)
    assert exp >= 0
print("300 test lớn: mọi cách khớp Cách 2")

400 test nhỏ: mọi cách khớp brute force


300 test lớn: mọi cách khớp Cách 2


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên chặn âm — khởi tạo đáp án bằng `-inf`

In [22]:
def sai_khong_chan_am(nums):
    best = float("-inf")                  # sai: nếu mọi bộ ba đều âm sẽ trả về số âm
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            for k in range(j + 1, len(nums)):
                best = max(best, (nums[i] - nums[j]) * nums[k])
    return best

print("sai :", sai_khong_chan_am([1, 2, 3]))                  # -3
print("đúng:", Solution().maximumTripletValue([1, 2, 3]))     # 0

sai : -3
đúng: 0


**Vì sao sai:** đề yêu cầu "nếu mọi bộ ba âm thì trả `0`". Khởi tạo `best = 0` (hoặc `max(0, …)`) giải quyết gọn.

### Lỗi 2: Dùng `max` và `min` toàn mảng, bỏ qua thứ tự `i < j < k`

In [23]:
def sai_max_min(nums):
    return max(0, (max(nums) - min(nums)) * max(nums))

print("sai :", sai_max_min([1, 2, 3]))                        # 6
print("đúng:", Solution().maximumTripletValue([1, 2, 3]))     # 0
print("sai :", sai_max_min([1, 10, 3, 4, 19]), "| đúng:", Solution().maximumTripletValue([1, 10, 3, 4, 19]))

sai : 6
đúng: 0
sai : 342 | đúng: 133


**Vì sao sai:** phần tử lớn nhất / nhỏ nhất có thể nằm **sai vị trí** (ví dụ `nums[i]` lớn nhất phải đứng **trước** `nums[j]`). Với `[1, 10, 3, 4, 19]`, `max - min = 18` nhưng `1` đứng trước `10` nên không dùng được.

### Lỗi 3: Prefix max / suffix max bao gồm cả `j`

In [24]:
def sai_pre_gom_j(nums):
    n = len(nums)
    best = 0
    for j in range(1, n - 1):
        left = max(nums[:j + 1])          # sai: gồm cả nums[j] (phải là nums[:j])
        right = max(nums[j:])             # sai: gồm cả nums[j] (phải là nums[j+1:])
        best = max(best, (left - nums[j]) * right)
    return best

print("sai :", sai_pre_gom_j([3, 5, 2, 1]))                    # dùng k = j nên cho kết quả quá lớn
print("đúng:", Solution().maximumTripletValue([3, 5, 2, 1]))

sai : 6
đúng: 3


**Vì sao sai:** `i < j < k` là bất đẳng thức **chặt**. Phần bên trái là `nums[:j]` và bên phải là `nums[j+1:]`. Nếu gồm `j`, ta có thể chọn `k = j`, một bộ ba không hợp lệ.

### Lỗi 4: Một lượt duyệt nhưng cập nhật sai thứ tự

In [25]:
def sai_thu_tu(nums):
    best, max_diff, imax = 0, 0, nums[0]
    for x in nums[1:]:
        imax = max(imax, x)               # sai: cập nhật imax trước khi dùng
        max_diff = max(max_diff, imax - x)
        best = max(best, max_diff * x)
    return best

print("sai :", sai_thu_tu([1, 2, 3]), sai_thu_tu([12, 6, 1, 2, 7]))
print("đúng:", Solution().maximumTripletValue([1, 2, 3]), Solution().maximumTripletValue([12, 6, 1, 2, 7]))

sai : 0 77
đúng: 0 77


**Vì sao sai:** ở bước `t`, `x` chỉ được làm `k` **trước** khi làm `j` hoặc `i` (vì `j < k`, `i < j`). Cập nhật `max_diff` bằng chính `x` rồi nhân luôn với `x` tức là dùng `j = k`.

Quy tắc nhớ: trong một lượt duyệt, cập nhật theo thứ tự **vai phía sau trước** (`k` → `j` → `i`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Biểu thức có dạng `f(trái) · g(giữa) · h(phải)` với `trái < giữa < phải` → **cố định phần tử giữa**, lấy tốt nhất bên trái (prefix) và bên phải (suffix).
- Khi các số đều dương, nhân với **số lớn nhất** bên phải là tham lam đúng (nếu hệ số khác dấu thì phải cẩn thận).
- Tối ưu `O(n³)` → `O(n)`: dựng prefix / suffix → rồi biến thành biến chạy `O(1)`; nhớ thứ tự cập nhật `k → j → i`.
- `n ≤ 100` thường cho phép `O(n³)` hoặc `O(n²)`; nhưng nên viết bản `O(n)` để chuẩn bị cho bài khó hơn (xem phần Biến thể).
- Luôn nghĩ đến "đáp án tối thiểu bằng 0" khi đề nói "trả 0 nếu không có giá trị dương".

## 15. Biến thể của bài

1. **LeetCode 2874 — Maximum Value of an Ordered Triplet II:** cùng đề, nhưng `n` lên tới `10⁵` (`O(n³)` và `O(n²)` đều quá chậm). Cách 3 / Cách 4 của notebook này giải được ngay.
2. **Tối đa `nums[i] * nums[j] * nums[k]`** (LeetCode 628 *Maximum Product of Three Numbers*): lưu ý số âm.
3. **Tối đa `nums[i] + nums[j] + nums[k]` với `nums[i] < nums[j] < nums[k]`** hay các biến thể có điều kiện giá trị.
4. **`nums[i] - nums[j] + nums[k]`:** cộng thay nhân (tối đa hóa hai phần độc lập).
5. **Tối đa `(nums[j] - nums[i])` với `i < j`:** LeetCode 121 *Best Time to Buy and Sell Stock* — chính là `max_diff` trong Cách 4.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ba vòng lặp | thử mọi `(i, j, k)` | `O(n³)` | `O(1)` | kiểm chứng; `n` rất nhỏ |
| 2. Hai vòng + suffix max | cố định `(i, j)`, `k` tốt nhất có sẵn | `O(n²)` | `O(n)` | bước chuyển, `n ≤ 10³` |
| 3. Prefix / suffix max | cố định `j` | `O(n)` | `O(n)` | rõ ràng, dễ chứng minh |
| 4. Một lượt | biến chạy `imax`, `max_diff` | `O(n)` | `O(1)` | **cách nộp bài**, tối ưu |
| 5. `accumulate` | prefix / suffix bằng thư viện | `O(n)` | `O(n)` | ngắn gọn khi nộp nhanh |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `list(accumulate([2, 1, 5, 3], max))`, `max(0, (3 - 7) * 4)`, `(8 - 2) * 5`, `len(list(combinations(range(5), 3)))`.

**Bài 2 (tính tay).** Với `nums = [3, 1, 4, 1, 5]`: lập bảng `j = 1, 2, 3` gồm `pre[j]`, `nums[j]`, `suf[j]` và giá trị `(pre - nums[j]) * suf`. Đáp án là gì?

**Bài 3 (điền TODO).** Hoàn thành `triplet_todo(nums)` (Cách 4).

In [26]:
def triplet_todo(nums):
    best = 0
    max_diff = 0
    imax = nums[0]
    for x in nums[1:]:
        # TODO 1: best = max(best, max_diff * x)
        # TODO 2: max_diff = max(max_diff, imax - x)
        # TODO 3: imax = max(imax, x)
        pass
    return best

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `triplet_prefix_suffix(nums)` (Cách 3) mà không nhìn code mẫu. Kiểm tra với ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `max_adjacent_gap_triplet(nums)`: giá trị lớn nhất của `nums[i] - nums[j] + nums[k]` với `i < j < k` (cộng thay vì nhân; **không** chặn dưới ở `0`). (b) Viết `best_buy_sell(prices)`: lợi nhuận lớn nhất `prices[j] - prices[i]` với `i < j` (trả `0` nếu không có lãi) — bài 121.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `accumulate([2, 1, 5, 3], max) = [2, 2, 5, 5]`; `max(0, -16) = 0`; `(8 - 2) * 5 = 30`; `C(5, 3) = 10`. Bài 2: `pre[1..3] = 3, 3, 4` và `suf[1..3] = 5, 5, 5`; tính từng giá trị `(pre - nums[j]) * suf` rồi lấy lớn nhất.
- **Mức 2 (vừa):** Bài 3: ba dòng đã ghi trong comment; nhớ thứ tự `k` → `j` → `i`. Bài 4: dựng `pre` từ trái sang phải, `suf` từ phải sang trái.
- **Mức 3 (sâu):** Bài 5(a): cố định `j`; muốn `nums[i]` lớn nhất bên trái và `nums[k]` lớn nhất bên phải; lúc này không cần chặn 0 nên đáp án là `max` thuần. Bài 5(b): giữ `lowest` (giá nhỏ nhất đã thấy) và cập nhật `best = max(best, price - lowest)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert list(accumulate([2, 1, 5, 3], max)) == [2, 2, 5, 5]
assert max(0, (3 - 7) * 4) == 0
assert (8 - 2) * 5 == 30
assert len(list(combinations(range(5), 3))) == 10

# Bài 2: nums = [3,1,4,1,5]
#  j=1: pre=3, nums=1, suf=5 -> (3-1)*5 = 10
#  j=2: pre=3, nums=4, suf=5 -> (3-4)*5 = -5
#  j=3: pre=4, nums=1, suf=5 -> (4-1)*5 = 15   <- lớn nhất
assert Solution().maximumTripletValue([3, 1, 4, 1, 5]) == 15
assert triplet_bruteforce([3, 1, 4, 1, 5]) == 15

# Bài 3
def triplet_todo_ans(nums):
    best = 0
    max_diff = 0
    imax = nums[0]
    for x in nums[1:]:
        best = max(best, max_diff * x)
        max_diff = max(max_diff, imax - x)
        imax = max(imax, x)
    return best

assert triplet_todo_ans([12, 6, 1, 2, 7]) == 77
assert triplet_todo_ans([1, 2, 3]) == 0

# Bài 4
def triplet_prefix_suffix_ans(nums):
    n = len(nums)
    pre = [0] * n
    cur = nums[0]
    for j in range(1, n):
        pre[j] = cur
        cur = max(cur, nums[j])
    suf = [0] * n
    cur = nums[-1]
    for j in range(n - 2, -1, -1):
        suf[j] = cur
        cur = max(cur, nums[j])
    return max([0] + [(pre[j] - nums[j]) * suf[j] for j in range(1, n - 1)])

assert triplet_prefix_suffix_ans([12, 6, 1, 2, 7]) == 77
assert triplet_prefix_suffix_ans([1, 10, 3, 4, 19]) == 133
assert triplet_prefix_suffix_ans([1, 2, 3]) == 0

# Bài 5(a): nums[i] - nums[j] + nums[k], không chặn 0
def max_adjacent_gap_triplet(nums):
    n = len(nums)
    pre, cur = [0] * n, nums[0]
    for j in range(1, n):
        pre[j] = cur
        cur = max(cur, nums[j])
    suf, cur = [0] * n, nums[-1]
    for j in range(n - 2, -1, -1):
        suf[j] = cur
        cur = max(cur, nums[j])
    return max(pre[j] - nums[j] + suf[j] for j in range(1, n - 1))

def brute_add(nums):
    return max(nums[i] - nums[j] + nums[k] for i, j, k in combinations(range(len(nums)), 3))

for seed_nums in ([12, 6, 1, 2, 7], [1, 2, 3], [5, 5, 5], [3, 1, 4, 1, 5, 9, 2, 6]):
    assert max_adjacent_gap_triplet(seed_nums) == brute_add(seed_nums)

# Bài 5(b)
def best_buy_sell(prices):
    lowest, best = prices[0], 0
    for p in prices[1:]:
        best = max(best, p - lowest)
        lowest = min(lowest, p)
    return best

assert best_buy_sell([7, 1, 5, 3, 6, 4]) == 5
assert best_buy_sell([7, 6, 4, 3, 1]) == 0
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cố định phần tử giữa; một lượt, thứ tự cập nhật `k → j → i`)

```python
from typing import List

class Solution:
    def maximumTripletValue(self, nums: List[int]) -> int:
        best = 0                 # đồng thời chặn âm
        max_diff = 0             # max(nums[i] - nums[j]) với i < j đã duyệt
        imax = nums[0]           # max các phần tử đã duyệt
        for x in nums[1:]:
            best = max(best, max_diff * x)        # x là nums[k]
            max_diff = max(max_diff, imax - x)    # x là nums[j]
            imax = max(imax, x)                   # x là nums[i] cho sau này
        return best
```

**Checklist**

- [ ] Hiểu `i < j < k` **chặt** (không dùng chung chỉ số)?
- [ ] Nếu mọi bộ ba âm thì trả `0` (khởi tạo `best = 0`)?
- [ ] Prefix max chỉ gồm phần tử **trước** `j`, suffix max chỉ gồm phần tử **sau** `j`?
- [ ] Thứ tự cập nhật trong một lượt: `k` trước, rồi `j`, rồi `i`?
- [ ] Đã thử `n = 3`, mảng tăng dần, giảm dần, toàn bằng nhau?
- [ ] Signature đúng `maximumTripletValue(self, nums: List[int]) -> int`?